# Hawaiian Newspaper OCR → Reference Audit → Translation

This notebook builds a pipeline for working with scanned **Hawaiian-language** newspapers:

1. **Load** a PDF of page images and render each page to a high-resolution image.
2. **OCR** the text with **Tesseract** (Latin/`eng` model — Tesseract has no Hawaiian model).
3. **Audit** a reference transcription you provide (e.g. from the Papakilo / Nūpepa database) against what is actually on the page — flagging where the reference disagrees with the page so you can correct the reference.
4. **Translate** the corrected Hawaiian text to English (free Google Translate by default; Claude optionally, if you have an API key).

### The page is the source of truth
The scanned page is the primary source. The reference text is a transcription that **may contain errors** — the goal of step 3 is to find those errors, not to trust the reference. Because OCR is itself imperfect, a flagged discrepancy is a *candidate* to check against the page image: some flags will be genuine reference errors, some will be OCR slips. You adjudicate.

Note: Tesseract has **no Hawaiian model**. The notebook uses the **Tongan model (`ton`)** — Tongan uses the same ʻokina (ʻ) and macrons (ā ē ī ō ū), so it approximates Hawaiian and can emit those marks (imperfectly). By default the audit folds diacritics out (`FOLD_DIACRITICS`), comparing base letters; flip that flag to audit diacritics too. Verify uncertain marks against the page image (or the optional Claude vision cell).

### Before you start
- Install the Tesseract engine (Step 0b).
- Put your newspaper PDF in this folder (default name `newspaper.pdf`).
- Have the reference transcription ready to paste in Step 5.
- Translation uses Google Translate by default (no key needed). To use Claude instead (higher quality on archaic Hawaiian), set `export ANTHROPIC_API_KEY=...` and `TRANSLATION_BACKEND = "claude"`.

Run the cells top to bottom.

## Step 0 — Install Python dependencies

Run once. These are the Python packages; the Tesseract engine itself is installed separately in Step 0b.

In [ ]:
# Install the Python packages. Safe to re-run; pip skips what's present.
%pip install --quiet pymupdf pytesseract opencv-python-headless pillow numpy deep-translator anthropic
print("Python dependencies installed.")

## Step 0b — Install the Tesseract engine

Tesseract is a system program, not a pip package, and there is **no Hawaiian model** for it — we use the Latin/English model (`eng`), which ships with Tesseract. Install the engine:

```
# macOS
brew install tesseract

# Debian / Ubuntu
sudo apt-get install tesseract-ocr
```

Run that in a terminal (or in a notebook cell with a leading `!`), then run the check below.

In [ ]:
import shutil, subprocess

exe = shutil.which("tesseract")
if not exe:
    print("Tesseract not found on PATH. Install it (see above), then restart the kernel.")
    print("If it's installed in a custom location, set TESSERACT_CMD in the config cell.")
else:
    langs = subprocess.run([exe, "--list-langs"], capture_output=True, text=True).stdout.split()
    print("tesseract:", exe)
    print("English model ('eng') present:", "eng" in langs)
    print("Note: Tesseract has no Hawaiian ('haw') model - using 'eng'; the audit folds out diacritics.")

## Step 1 — Configuration

Edit the values below to point at your files and pick your options.

In [ ]:
import os
import re
import difflib
from pathlib import Path

import numpy as np
from PIL import Image

# ---------------------------------------------------------------------------
# Configuration — edit these
# ---------------------------------------------------------------------------
PDF_PATH       = "kkh_18360720.pdf"      # the scanned newspaper PDF (place it in this folder)
REFERENCE_PATH = "reference.txt"      # optional file fallback for the reference text
OUTPUT_DIR     = "output"             # where extracted text / reports / translations are written

RENDER_DPI = 300                      # higher = sharper images = better OCR (but slower)
PREPROCESS = True                     # apply grayscale / threshold / deskew before OCR

OCR_LANG      = "ton"                 # Tongan model: best available match - emits the ʻokina (and macrons). Try "ton+eng" or "mri". (No Hawaiian model exists.)
FOLD_DIACRITICS = True                # audit ignores ʻokina/macrons (robust). Set False to audit diacritics too - useful now that OCR is "ton".
TESSERACT_CMD = ""                    # optional: full path to the tesseract binary if it's not on PATH
FIX_ORIENTATION = True                # rotation guard: OSD-rotate pages upright (90/180/270)
N_COLUMNS       = "auto"             # "auto" = detect the count per page (variable); set an int to force it; 1 = no split
MAX_COLUMNS     = 8                   # auto mode never detects more than this many columns
TESS_PSM        = 4                   # per-column page-seg mode (4 = single column of variable sizes)

# Translation
TRANSLATION_BACKEND = "google"        # "google" (free, no key), "claude" (needs ANTHROPIC_API_KEY), or "auto"
CLAUDE_MODEL = "claude-opus-4-8"      # translation model (only used if backend is claude)
SOURCE_LANG = "haw"                   # Hawaiian (used by the Google backend)

# Set in Step 5 (paste cell); kept here so the audit cell never hits a NameError.
REFERENCE_TEXT = ""

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print("Config loaded. Output dir:", os.path.abspath(OUTPUT_DIR))

## Step 2 — Load the PDF and render pages to images

We render each page at `RENDER_DPI` with PyMuPDF. Rendering (rather than extracting embedded images) works for both scanned and vector PDFs.

In [ ]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path, dpi=300):
    # Render each page of a PDF to a PIL RGB image.
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(
            f"Could not find '{pdf_path}'. Put your PDF in this folder or update PDF_PATH."
        )
    doc = fitz.open(pdf_path)
    zoom = dpi / 72.0  # 72 is the PDF base DPI
    mat = fitz.Matrix(zoom, zoom)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat, alpha=False)
        images.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    doc.close()
    return images

pages = pdf_to_images(PDF_PATH, dpi=RENDER_DPI)
print(f"Rendered {len(pages)} page(s) at {RENDER_DPI} DPI.")

# Preview the first page (downscaled for display)
preview = pages[0].copy()
preview.thumbnail((700, 900))
preview

## Step 3 — Image cleanup (binarization + deskew helpers)

Defines the image helpers used by OCR: `to_binary` (grayscale → denoise → adaptive threshold, giving clean black-on-white text) and `deskew` (corrects small tilt). Deskew is applied **per column** during OCR, not to the whole page — a global deskew on a multi-column page is unreliable and can rotate text the wrong way. Set `PREPROCESS = False` to OCR the raw grayscale instead.

In [ ]:
import cv2

def to_binary(pil_img):
    # grayscale + denoise + adaptive threshold -> text black (0) on white (255)
    img = np.array(pil_img.convert("L"))
    img = cv2.fastNlMeansDenoising(img, h=10)
    return cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15
    )

def deskew(bw):
    # Correct small tilt (a few degrees) of a SINGLE text block / column.
    coords = np.column_stack(np.where(bw < 255)).astype(np.float32)
    if len(coords) == 0:
        return bw
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) <= 0.5 or abs(angle) > 20:   # ignore tiny + implausible angles
        return bw
    h, w = bw.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(bw, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=255)

def preprocess(pil_img):
    # Binarized whole page (used for the preview below).
    return Image.fromarray(to_binary(pil_img))

if PREPROCESS:
    prev = preprocess(pages[0]).copy(); prev.thumbnail((700, 900)); display(prev)
else:
    print("Preprocessing disabled (PREPROCESS = False) - OCR will run on the raw grayscale page.")

## Step 4 — OCR with Tesseract (column + rotation guards)

Newspaper pages break two of Tesseract's assumptions, so two guards handle them:

- **Rotation guard (`FIX_ORIENTATION`)** — Tesseract OSD detects 90°/180°/270° page rotation and rotates the scan upright before OCR. (Small tilt is handled by per-column `deskew`; a *localized* rotated block — e.g. a caption set sideways next to an engraving — is **not** auto-corrected.)
- **Column guard (`N_COLUMNS`)** — newspapers vary in column count, so by default (`N_COLUMNS = "auto"`) the count is **detected per page** from the page's **vertical ink profile** (measured below the masthead, with engravings and full-width rules filtered out so they can't bridge a gutter). The profile is scored for 2…`MAX_COLUMNS` columns and the split is taken at the **knee** — the most columns whose every gutter is still a deep whitespace valley rather than a cut through solid text. Each column is then OCR'd on its own, left-to-right. Set `N_COLUMNS` to an **integer** to force that many columns, or `1` to disable splitting. (This paper's three text pages auto-detect as **3 columns**; the illustrated front page, where an engraving fills part of a column, detects as **2** — set `N_COLUMNS = 3` if you want it split anyway.)

The cell first draws the detected boundaries on page 1 (and prints the detected count) so you can sanity-check the split, then OCRs each column with `--psm` = `TESS_PSM` (4 = single column of variable sizes) using the Tongan model. If the auto split looks wrong, force `N_COLUMNS` to an integer, or tune the detection knobs in this cell (`DEPTH_THRESH`, `EVEN_THRESH`, `SOLID_RUN`, `COL_TOP_SKIP`).

In [ ]:
import pytesseract
import cv2
from pytesseract import Output

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

# --- column-detection tunables (used by detect_columns below) ---
COL_TOP_SKIP = 0.18    # ignore the top band (full-width masthead / dateline) when finding gutters
COL_BOT_SKIP = 0.05    # ignore the very bottom band (footer / page rule)
SMOOTH_FRAC  = 0.004   # light smoothing of the ink profile (bridges intra-word gaps, keeps gutters)
DEPTH_THRESH = 0.72    # a real gutter is a near-empty valley >=72% below its flanking columns
EVEN_THRESH  = 0.18    # reject splits whose narrowest column < 18% of the widest
SOLID_RUN    = 0.15    # drop rows with a solid ink run >= this fraction of width (engravings / rules)


def fix_orientation(pil_img):
    # ROTATION GUARD: Tesseract OSD rotates a sideways/upside-down page upright.
    if not FIX_ORIENTATION:
        return pil_img
    try:
        osd = pytesseract.image_to_osd(pil_img, output_type=Output.DICT)
        rot = int(osd.get("rotate", 0)); conf = float(osd.get("orientation_conf", 0))
        if rot % 360 != 0 and conf >= 1.0:
            print(f"  orientation: rotating {rot} deg (conf {conf:.1f})")
            return pil_img.rotate(-rot, expand=True, fillcolor=(255, 255, 255))
    except Exception as e:
        print("  (OSD skipped:", e, ")")
    return pil_img


def _median(a):
    a = np.sort(np.asarray(a, dtype=np.float64))
    if a.size == 0:
        return 0.0
    m = a.size // 2
    return float(a[m]) if a.size % 2 else float(0.5 * (a[m - 1] + a[m]))


def _ink_profile(gray):
    # Vertical ink projection over the body band, excluding rows that cross a solid
    # graphic / rule (so an engraving or full-width line can't bridge a gutter).
    # Using raw ink (not Tesseract word boxes) keeps detection stable across DPI.
    H, W = gray.shape
    ink = (gray < 128).astype(np.uint8)
    L = max(8, int(W * SOLID_RUN))
    ker = cv2.getStructuringElement(cv2.MORPH_RECT, (L, 1))
    solid_row = cv2.erode(ink, ker).any(axis=1)
    lo_r, hi_r = int(COL_TOP_SKIP * H), int((1.0 - COL_BOT_SKIP) * H)
    keep = ~solid_row[lo_r:hi_r]
    rows = np.arange(lo_r, hi_r)[keep]
    if rows.size < 10:                       # almost everything looked solid -> use the whole band
        rows = np.arange(lo_r, hi_r)
    cov = ink[rows, :].sum(axis=0).astype(np.float64)
    k = max(3, int(W * SMOOTH_FRAC))
    return np.convolve(cov, np.ones(k) / k, mode="same")


def detect_columns(gray, n_force=None):
    # COLUMN GUARD: auto-detect the number of text columns (variable per page).
    # A true N-column page has N-1 deep whitespace gutters in the vertical ink
    # profile. We score splits into 2..MAX_COLUMNS by the WORST gutter's local
    # depth (vs its two flanking column peaks); as n passes the real column count
    # the extra cut must fall on solid text and that depth collapses -> we keep the
    # largest n whose every gutter still clears DEPTH_THRESH. n_force, if given,
    # snaps exactly that many columns instead. Returns {"n","cuts","bounds","diag"}.
    H, W = gray.shape

    def whole(d):
        return {"n": 1, "cuts": [], "bounds": [(0, W)], "diag": d}

    cov = _ink_profile(gray)
    pos = cov[cov > 0]
    if pos.size == 0:
        return whole("blank-page -> n=1")
    nz = np.where(cov > 0.15 * _median(pos))[0]   # text span, trimming side margins
    if nz.size == 0:
        return whole("empty-coverage -> n=1")
    lo, hi = int(nz[0]), int(nz[-1])
    span = hi - lo
    if span < 0.25 * W:
        return whole("narrow-text-span -> n=1")

    interior = np.arange(lo + 1, hi)
    is_min = (cov[interior] <= cov[interior - 1]) & (cov[interior] <= cov[interior + 1])
    minima = interior[is_min]

    def local_depth(x, halfwin):
        a = max(lo, x - halfwin)
        b = min(hi, x + halfwin)
        left_peak = cov[a:x + 1].max() if x > a else cov[x]
        right_peak = cov[x:b + 1].max() if b > x else cov[x]
        flank = 0.5 * (left_peak + right_peak)
        if flank <= 0:
            return 0.0
        return max(0.0, (flank - cov[x]) / flank)     # local relative depth in [0,1]

    def cuts_for_n(n):
        # n-1 cuts at the deepest valleys, min-separation ~ a column width apart.
        sep = max(1, int(span / n * 0.45))
        cand = [int(x) for x in minima if lo + sep <= x <= hi - sep]
        cand.sort(key=lambda x: cov[x])               # deepest valleys first
        chosen = []
        for x in cand:
            if all(abs(x - c) > sep for c in chosen):
                chosen.append(x)
            if len(chosen) == n - 1:
                break
        return sorted(chosen)

    nmax = min(MAX_COLUMNS, 8)

    # score every candidate n by (worst local gutter depth, column-width evenness).
    scores = {}
    for n in range(2, nmax + 1):
        cuts = cuts_for_n(n)
        if len(cuts) < n - 1:
            continue
        halfwin = max(1, int(span / n * 0.5))
        worst = min(local_depth(x, halfwin) for x in cuts)
        edges = [lo] + cuts + [hi]
        widths = np.diff(edges)
        even = float(widths.min() / widths.max())
        scores[n] = (worst, even, cuts)

    # config override: force exactly n columns (skips auto selection).
    if n_force is not None and int(n_force) >= 2:
        nf = min(int(n_force), nmax)
        cuts = cuts_for_n(nf)
        if len(cuts) == nf - 1:
            edges = [lo] + cuts + [hi]
            return {"n": nf, "cuts": [int(c) for c in cuts],
                    "bounds": [(int(edges[i]), int(edges[i + 1])) for i in range(len(edges) - 1)],
                    "diag": "forced n=%d" % nf}
        return whole("forced n=%d unsatisfiable -> n=1" % nf)

    # auto: keep the largest n whose worst gutter clears the depth + evenness gates.
    best_n = 1
    for n in range(2, nmax + 1):
        if n not in scores:
            continue
        worst, even, _ = scores[n]
        if worst >= DEPTH_THRESH and even >= EVEN_THRESH:
            best_n = n
    if best_n <= 1:
        return whole("no-gutter-clears-depth>=%.2f -> n=1" % DEPTH_THRESH)

    cuts = scores[best_n][2]
    edges = [lo] + cuts + [hi]
    bounds = [(int(edges[i]), int(edges[i + 1])) for i in range(len(edges) - 1)]
    diag = "auto n=%d span=%d " % (best_n, span) + " ".join(
        "[%d:%.2f]" % (m, scores[m][0]) for m in sorted(scores))
    return {"n": best_n, "cuts": [int(c) for c in cuts], "bounds": bounds, "diag": diag}


def column_bounds(src, n):
    # Map the N_COLUMNS config to actual column x-bounds:
    #   "auto"  -> detect the count per page (variable),
    #   int >=2 -> force exactly that many columns,
    #   1       -> a single whole-page crop (splitting disabled).
    W, H = src.size
    if n == 1:
        return [(0, W)]
    gray = np.asarray(src.convert("L"))
    n_force = None if (isinstance(n, str) and n.lower() == "auto") else int(n)
    if n_force is not None and n_force < 2:
        return [(0, W)]
    return detect_columns(gray, n_force=n_force)["bounds"]


PAD = 8  # px kept around each column crop

def ocr_image(pil_img):
    img = fix_orientation(pil_img)
    src = preprocess(img) if PREPROCESS else img.convert("L")
    W, H = src.size
    cols = column_bounds(src, N_COLUMNS)
    arr = np.array(src)
    cfg = f"--oem 1 --psm {TESS_PSM}"
    parts = []
    for a, b in cols:
        a, b = max(0, a - PAD), min(W, b + PAD)
        crop = arr[:, a:b]
        if PREPROCESS:
            crop = deskew(crop)
        txt = pytesseract.image_to_string(Image.fromarray(crop), lang=OCR_LANG, config=cfg)
        if txt.strip():
            parts.append(txt.strip())
    return "\n\n".join(parts), len(cols)


def show_columns(pil_img):
    # Visual check: draw the detected column boundaries (red) on the page.
    from PIL import ImageDraw
    img = fix_orientation(pil_img)
    src = preprocess(img) if PREPROCESS else img.convert("L")
    cols = column_bounds(src, N_COLUMNS)
    print(f"  detected {len(cols)} column(s): {cols}")
    ov = img.convert("RGB"); dr = ImageDraw.Draw(ov)
    for a, b in cols:
        dr.line([(a, 0), (a, ov.height)], fill=(255, 0, 0), width=6)
        dr.line([(b, 0), (b, ov.height)], fill=(255, 0, 0), width=6)
    ov.thumbnail((700, 950)); return ov

# Sanity-check the column split on the first page before OCRing everything:
display(show_columns(pages[0]))

ocr_pages = []
for i, page in enumerate(pages, start=1):
    text, n_cols = ocr_image(page)
    ocr_pages.append(text)
    print(f"[page {i}/{len(pages)}] {n_cols} column(s), {len(text)} chars")

ocr_text = "\n\n".join(ocr_pages)

ocr_out = Path(OUTPUT_DIR) / "extracted_ocr.txt"
ocr_out.write_text(ocr_text, encoding="utf-8")
print(f"\nSaved raw OCR to {ocr_out}\n")
print("----- preview -----")
print(ocr_text[:1500])


## Step 5 — provide the reference text to audit

Paste the reference transcription (e.g. the text shown on the Papakilo article page) into `REFERENCE_TEXT` below, then run it. This is the text we will check for errors — re-run with new text for each article.

If you correct the reference after reviewing the discrepancies, paste the fixed text back here and re-run the audit.

In [ ]:
# === Paste the reference transcription to audit here, then run this cell. ===
# Replace the placeholder line between the ''' markers. Re-run for each new article.
REFERENCE_TEXT = '''
NO NOA MA. Olelo mai la ke Akua ia Noa, i mai la, E hele aku oe iwaho o ka halelana, o oe, me kau wahine a me kau niau keikikane, a me na wahine a kau mau keikikane me oe. E lawe pu mai me oe

i na mea ola a pau me oe, i na kino a pau o na manu, me na holoholona, a me na mea kolo a pau e kolo ana maluna o ka honua; i hanau nui ai lakou ma ka honua, i hua mai a mahuahua maluna o ka honua. Hele aku la o Noa me

kana mau keiki, a me kana wahine, a me na wahine a kank mau keikikane me ia. A o na holoholona a pau, na mea kolo a pau, me na manu a pau, a me na mea hele a pau maluna o ka honua, ma ko lakou ano iho, hele aku la lakou iwaho o ka halelana. Hana iho la o Noa i ke kuahu 110 lehova; lalau aku la ia i kekahi o ua holoholona maemae, a o na ihanu maemae a pau, a kaumaha aku la i mohaikuni maluna o ke kuahu.
'''

print(len(REFERENCE_TEXT.strip()), "chars of reference text loaded.")

### Audit the reference against the page

Neither side is assumed correct. This step:

- **Normalizes** both texts (unifies ʻokina variants, collapses whitespace). When `FOLD_DIACRITICS = True` (default) it also **folds out the ʻokina and macrons on both sides** before comparing, so partial OCR capture doesn't create false diacritic flags. Set `FOLD_DIACRITICS = False` to audit diacritics too (the `ton` model can read them).
- **Aligns** the reference against the page OCR word-by-word and **flags every discrepancy** — `DIFFERS`, `PAGE HAS / REFERENCE LACKS` (reference may be missing text), and `REFERENCE HAS / PAGE LACKS` (reference may have added text not on the page).
- Writes a numbered discrepancy report and an annotated copy of the reference, and shows a visual diff.

Expect some flags to be OCR noise — verify each against the page image before changing the reference.

In [ ]:
# Resolve the reference text: pasted REFERENCE_TEXT wins, then reference.txt fallback.
if REFERENCE_TEXT.strip():
    reference_text = REFERENCE_TEXT
elif os.path.exists(REFERENCE_PATH):
    reference_text = Path(REFERENCE_PATH).read_text(encoding="utf-8")
else:
    reference_text = ""
    print("No reference text. Paste it into REFERENCE_TEXT in the cell above to audit it.")

import unicodedata

# ---------------------------------------------------------------------------
# Hawaiian-aware normalization (unify ʻokina variants, collapse whitespace).
# ---------------------------------------------------------------------------
OKINA = "ʻ"  # U+02BB
APOSTROPHE_VARIANTS = ["'", "’", "‘", "`", "´", "ʼ"]

def normalize(text):
    for ch in APOSTROPHE_VARIANTS:
        text = text.replace(ch, OKINA)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def fold(tok):
    # Drop diacritics (ʻokina, macrons/accents) so partial OCR diacritic capture
    # doesn't create false flags. Used only when FOLD_DIACRITICS is True.
    for o in APOSTROPHE_VARIANTS + [OKINA]:
        tok = tok.replace(o, "")
    tok = unicodedata.normalize("NFKD", tok)
    return "".join(c for c in tok if not unicodedata.combining(c))

def audit_reference(reference, ocr):
    # Compare reference (a) vs page OCR (b) on diacritic-folded tokens, but report
    # the original reference/page text. Neither side is assumed correct.
    ref = normalize(reference).split()
    pg = normalize(ocr).split()
    _fold = fold if FOLD_DIACRITICS else (lambda t: t)
    ref_f = [_fold(t) for t in ref]
    pg_f = [_fold(t) for t in pg]
    sm = difflib.SequenceMatcher(a=ref_f, b=pg_f, autojunk=False)
    discrepancies, annotated = [], []
    CTX = 4
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            annotated.extend(ref[i1:i2])
            continue
        ref_span = " ".join(ref[i1:i2])
        pg_span = " ".join(pg[j1:j2])
        before = " ".join(ref[max(0, i1 - CTX):i1])
        after = " ".join(ref[i2:i2 + CTX])
        discrepancies.append({
            "type": tag,
            "reference": ref_span,
            "page_ocr": pg_span,
            "context": f"...{before} [HERE] {after}...",
        })
        if tag == "replace":
            annotated.append(f"⟦REF:{ref_span} | PAGE:{pg_span}⟧")
        elif tag == "delete":
            annotated.append(f"⟦REF-ONLY:{ref_span}⟧")
        elif tag == "insert":
            annotated.append(f"⟦PAGE-ONLY:{pg_span}⟧")
    return discrepancies, " ".join(annotated), round(sm.ratio(), 4)

KIND = {
    "replace": "DIFFERS",
    "delete": "REFERENCE HAS / PAGE LACKS",
    "insert": "PAGE HAS / REFERENCE LACKS",
}

if reference_text.strip():
    discrepancies, annotated_ref, similarity = audit_reference(reference_text, ocr_text)
    note = "diacritics folded" if FOLD_DIACRITICS else "diacritics compared"
    print(f"Reference vs page-OCR similarity ({note}): {similarity}")
    print(f"{len(discrepancies)} discrepancy span(s) flagged for review.\n")

    lines = []
    for n, d in enumerate(discrepancies, 1):
        lines.append(
            f"[{n}] {KIND[d['type']]}\n"
            f"    reference: {d['reference'] or '(none)'}\n"
            f"    page OCR : {d['page_ocr'] or '(none)'}\n"
            f"    near     : {d['context']}\n"
        )
    report = "\n".join(lines) if lines else "No discrepancies found."
    (Path(OUTPUT_DIR) / "discrepancies.txt").write_text(report, encoding="utf-8")
    (Path(OUTPUT_DIR) / "reference_annotated.txt").write_text(annotated_ref, encoding="utf-8")
    print(report[:2000])
    print("\nSaved discrepancies.txt and reference_annotated.txt to", OUTPUT_DIR)
else:
    annotated_ref = ""
    discrepancies = []

In [ ]:
# Visual diff: reference (left) vs page OCR (right). Highlighted spans are the discrepancies.
from IPython.display import HTML, display

def audit_diff(reference, ocr, full=False):
    a = normalize(reference).split("\n")
    b = normalize(ocr).split("\n")
    differ = difflib.HtmlDiff(wrapcolumn=80)
    maker = differ.make_file if full else differ.make_table
    return maker(a, b, fromdesc="Reference (under review)", todesc="Page OCR (Tesseract)",
                 context=True, numlines=2)

if reference_text.strip():
    (Path(OUTPUT_DIR) / "diff.html").write_text(
        audit_diff(reference_text, ocr_text, full=True), encoding="utf-8")
    display(HTML(audit_diff(reference_text, ocr_text)))
    print("Full diff saved to", Path(OUTPUT_DIR) / "diff.html")
else:
    print("Provide reference text to see a diff.")

In [ ]:
# Choose the text to carry forward to translation.
# Default = your reference transcription. After you review the discrepancies and fix
# the reference (re-run the paste + audit cells), this updates automatically. Or set
# final_text to something else here (e.g. the OCR text, or a Claude-adjudicated version).
final_text = reference_text if reference_text.strip() else normalize(ocr_text)
print(f"final_text set ({len(final_text)} chars). Edit the reference and re-run to update it.")

### Step 5b — (Optional) Claude-assisted adjudication

If you have an `ANTHROPIC_API_KEY`, Claude can weigh the reference against the page OCR and propose a corrected reference, restoring ʻokina and kahakō. (Better still, use the Claude vision cell at the end to read the actual image.) Uncomment the bottom block to run.

In [ ]:
def claude_adjudicate(reference, ocr, model=CLAUDE_MODEL):
    import anthropic
    client = anthropic.Anthropic()
    system = (
        "You are auditing a Hawaiian (ʻŌlelo Hawaiʻi) newspaper transcription. "
        "(A) is a REFERENCE transcription that may contain errors. "
        "(B) is OCR read directly from the scanned page. Neither is fully reliable. "
        "Using both as evidence, produce the most accurate Hawaiian text, correcting errors "
        "in the reference where the page clearly disagrees. Restore ʻokina (ʻ) and kahakō "
        "(ā ē ī ō ū). Return ONLY the corrected Hawaiian text, no commentary."
    )
    user = f"=== A: REFERENCE ===\n{reference}\n\n=== B: PAGE OCR ===\n{ocr}"
    resp = client.messages.create(
        model=model, max_tokens=16000, system=system,
        messages=[{"role": "user", "content": user}],
    )
    return next((b.text for b in resp.content if b.type == "text"), "")

# Uncomment to run and carry the result forward to translation:
# if reference_text.strip() and os.environ.get("ANTHROPIC_API_KEY"):
#     final_text = claude_adjudicate(reference_text, ocr_text)
#     (Path(OUTPUT_DIR) / "corrected_claude.txt").write_text(final_text, encoding="utf-8")
#     print(final_text[:1500])
# else:
#     print("Set ANTHROPIC_API_KEY and provide reference text to use Claude adjudication.")

## Step 6 — Translate Hawaiian → English

Default backend is **Google Translate** (`deep-translator`) — free, no API key, needs internet. It supports Hawaiian but is rough on archaic newspaper text.

If you ever get an `ANTHROPIC_API_KEY`, set `TRANSLATION_BACKEND = "claude"` in the config for much better 19th-/early-20th-century Hawaiian translation. Long text is chunked automatically either way.

In [ ]:
# ---------------------------------------------------------------------------
# Translate Hawaiian -> English. Translates `final_text` (see the cell above).
# ---------------------------------------------------------------------------
def _chunk(text, size=4000):
    # Split text into <=size-char chunks on line boundaries.
    parts, buf = [], ""
    for line in text.split("\n"):
        if len(buf) + len(line) + 1 > size and buf:
            parts.append(buf); buf = ""
        buf += line + "\n"
    if buf.strip():
        parts.append(buf)
    return parts

def translate_google(text, source=SOURCE_LANG):
    # Free, no key, needs internet. Falls back to auto-detect if the code is rejected.
    from deep_translator import GoogleTranslator
    try:
        tr = GoogleTranslator(source=source, target="en")
        return "\n".join(tr.translate(c) for c in _chunk(text, size=4500))
    except Exception:
        tr = GoogleTranslator(source="auto", target="en")
        return "\n".join(tr.translate(c) for c in _chunk(text, size=4500))

def translate_claude(text, model=CLAUDE_MODEL):
    # Optional upgrade; requires ANTHROPIC_API_KEY.
    import anthropic
    client = anthropic.Anthropic()
    system = (
        "You are an expert translator of ʻŌlelo Hawaiʻi (Hawaiian) into English, "
        "specializing in 19th- and early-20th-century Hawaiian-language newspapers. "
        "Translate faithfully and naturally. Keep proper nouns and place names in Hawaiian. "
        "If a passage is garbled or uncertain, translate your best reading and mark it [unclear]. "
        "Return ONLY the English translation."
    )
    outs = []
    for chunk in _chunk(text, size=4000):
        resp = client.messages.create(
            model=model, max_tokens=16000, system=system,
            messages=[{"role": "user", "content": chunk}],
        )
        outs.append(next((b.text for b in resp.content if b.type == "text"), ""))
    return "\n".join(outs)

def translate(text):
    backend = TRANSLATION_BACKEND
    if backend == "auto":
        backend = "claude" if os.environ.get("ANTHROPIC_API_KEY") else "google"
    if backend == "claude" and not os.environ.get("ANTHROPIC_API_KEY"):
        print("No ANTHROPIC_API_KEY found - using Google Translate instead.")
        backend = "google"
    print(f"Translating with: {backend}")
    return translate_claude(text) if backend == "claude" else translate_google(text)

english = translate(final_text)
en_out = Path(OUTPUT_DIR) / "translation_en.txt"
en_out.write_text(english, encoding="utf-8")
print("Saved English translation to", en_out, "\n")
print("----- preview -----")
print(english[:1500])

In [ ]:
# Side-by-side view of the carried-forward Hawaiian and the English translation.
import html as _html
from IPython.display import HTML

def side_by_side(haw, eng):
    h = _html.escape(haw); e = _html.escape(eng)
    cell = "style='vertical-align:top;width:50%;padding:8px;border:1px solid #eee'"
    return HTML(
        "<table style='width:100%'><tr>"
        + f"<td {cell}><b>Hawaiian (carried forward)</b><pre style='white-space:pre-wrap'>{h}</pre></td>"
        + f"<td {cell}><b>English</b><pre style='white-space:pre-wrap'>{e}</pre></td>"
        + "</tr></table>"
    )

side_by_side(final_text, english)

### (Optional) Cross-check the page with Claude vision

Claude can read a page image directly — the best way to adjudicate a flagged discrepancy, since it looks at the actual page rather than the OCR. Requires `ANTHROPIC_API_KEY`.

In [ ]:
import base64, io

def claude_vision_ocr(pil_img, model=CLAUDE_MODEL):
    import anthropic
    client = anthropic.Anthropic()
    img = pil_img.copy(); img.thumbnail((1500, 1500))   # keep the request small
    buf = io.BytesIO(); img.save(buf, format="PNG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode("utf-8")
    resp = client.messages.create(
        model=model, max_tokens=8000,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}},
            {"type": "text", "text": (
                "Transcribe ALL Hawaiian text on this newspaper page exactly, "
                "including ʻokina (ʻ) and kahakō (ā ē ī ō ū). Return only the transcription."
            )},
        ]}],
    )
    return next((b.text for b in resp.content if b.type == "text"), "")

# Uncomment to run on the first page:
# if os.environ.get("ANTHROPIC_API_KEY"):
#     print(claude_vision_ocr(pages[0])[:1500])
# else:
#     print("Set ANTHROPIC_API_KEY to use the Claude vision cross-check.")

## Outputs & next steps

Everything is written to the `output/` folder:

| File | What it is |
|------|------------|
| `extracted_ocr.txt` | Raw Tesseract OCR text from the page images |
| `discrepancies.txt` | Every place the reference disagrees with the page OCR — your review list |
| `reference_annotated.txt` | The reference with each discrepancy marked inline (`⟦REF:… \| PAGE:…⟧`) |
| `diff.html` | Side-by-side visual diff: reference vs page OCR (open in a browser) |
| `translation_en.txt` | English translation of the text you carried forward |

**Workflow**
1. Run the audit, then review `discrepancies.txt` / the diff against the page image.
2. Fix the genuine errors in your reference text, paste it back into Step 5, and re-run the audit until only OCR-noise flags remain.
3. Translate the corrected text.

**Tuning tips**
- If OCR is weak (lots of false flags): raise `RENDER_DPI` (e.g. 400), toggle `PREPROCESS`, or change `TESS_PSM` (try 6 for single-column clippings), or check the column/rotation guards (`N_COLUMNS`, `FIX_ORIENTATION`).
- For mixed Hawaiian/English pages set `OCR_LANG = "haw+eng"`.
- To adjudicate against the actual page automatically, use the optional Claude vision cell (needs an API key).